In [ ]:
import os
os.environ['OPENBLAS_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Iterator, List, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd
import pickle

import xgboost as xgb
import lightgbm as lgb
from lightgbm import LGBMClassifier
import shap

In [ ]:
from config import dir_config, random_seed

compiled_dir = dir_config.data.compiled
processed_dir = dir_config.data.processed

model_dir = Path(dir_config.data.base) / "models"
model_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# from src.hyperparameter_optimization import lgb_hyperparameter_fitting

In [ ]:
target_type = 'apnea'
target_future_step = 2
features = 'all'

max_feature_lag = 15
with_pca = False
with_metadata = True


# Load data

In [ ]:
if with_pca:
    filename = f"apnea_detection_{5*max_feature_lag}secs_lag_features_pc_{features}_{5*target_future_step}secs_future_clean_metadata_{str(with_metadata).lower()}"
else:
    filename = f"apnea_detection_{5*max_feature_lag}secs_lag_features_{features}_{5*target_future_step}secs_future_clean_metadata_{str(with_metadata).lower()}"

apnea_data = pickle.load(open(Path(processed_dir) / f"{filename}.pkl", "rb"))

In [ ]:
filename

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, classification_report, confusion_matrix

def evaluate_model(model, X, y, threshold=0.5):
    y_pred_proba = model.predict_proba(X)[:, 1]
    y_pred = (y_pred_proba >= threshold).astype(int)
    auc_roc = roc_auc_score(y, y_pred_proba)
    auc_pr = average_precision_score(y, y_pred_proba)
    f1 = f1_score(y, y_pred)
    #
    classification = classification_report(y, y_pred)
    confusion = confusion_matrix(y, y_pred)
    return {'auc_roc': auc_roc, 'auc_pr': auc_pr, 'f1': f1, 'classification_report': classification, 'confusion_matrix': confusion}

## Train/Val/Test Split

In [ ]:
train_X, train_y = pd.DataFrame(apnea_data['train_X']), pd.Series(apnea_data['train_y'])
val_X, val_y = pd.DataFrame(apnea_data['val_X']), pd.Series(apnea_data['val_y'])
test_X, test_y = pd.DataFrame(apnea_data['test_X']), pd.Series(apnea_data['test_y'])
fold_indices = apnea_data['fold_indices']

In [ ]:
train_X.shape, val_X.shape, test_X.shape

In [ ]:
# lgb_hyperparameter_fitting imported from src.hyperparameter_optimization

### LightGBM

In [ ]:
# model, cv_aucpr, study = lgb_hyperparameter_fitting(train_X, train_y, fold_indices, n_trials=50, use_smote=False)

# print("\n" + "="*70)
# print("TUNED MODEL EVALUATION: No-Lag Features")
# print("="*70)

# results = {}
# for split_name, X, y in [
#     ("Train", train_X, train_y),
#     ("Val  ", val_X, val_y),
# ]:
#     m = evaluate_model(model, X, y)
#     results[split_name.strip()] = m
#     print(f"\n{split_name} Set:")
#     print(f"  AUC-ROC={m['auc_roc']:.3f}  AUC-PR={m['auc_pr']:.3f}  F1={m['f1']:.3f}")
#     print("  Classification Report:\n", m['classification_report'])

# # Check for overfitting
# train_aucpr = results['Train']['auc_pr']
# val_aucpr = results['Val']['auc_pr']
# train_val_gap = train_aucpr - val_aucpr
# print("\n" + "-"*70)
# print(f"OVERFITTING DIAGNOSTICS:")
# print(f"  Train-Val gap:  {train_val_gap:.3f} (⚠ if > 0.20)")
# print(f"  CV AUC-PR:      {cv_aucpr:.3f} (expected close to Val)")
# print("-"*70 + "\n")

# # Save model (use test scores as reference)
# import joblib
# model_save_path = Path(processed_dir) / "model" / f"lgb_model_{filename}.joblib"
# model_save_path.parent.mkdir(parents=True, exist_ok=True)
# joblib.dump({
#     "model": model,
#     "cv_auc_pr": cv_aucpr,
#     "val_evaluation": results['Val'],
#     "study": study
# }, model_save_path)
# print(f"Saved to: {model_save_path}")

In [ ]:
filename

In [ ]:
from src.hyperparameter_optimization_nosmote import lgb_hyperparameter_fitting
model, cv_aucpr, study = lgb_hyperparameter_fitting(train_X, train_y, fold_indices, n_trials=50, random_seed=random_seed)

print("\n" + "="*70)
print("TUNED MODEL EVALUATION: No-Lag Features")
print("="*70)

results = {}
for split_name, X, y in [
    ("Train", train_X, train_y),
    ("Val  ", val_X, val_y),
]:
    m = evaluate_model(model, X, y)
    results[split_name.strip()] = m
    print(f"\n{split_name} Set:")
    print(f"  AUC-ROC={m['auc_roc']:.3f}  AUC-PR={m['auc_pr']:.3f}  F1={m['f1']:.3f}")
    print("  Classification Report:\n", m['classification_report'])

# Check for overfitting
train_aucpr = results['Train']['auc_pr']
val_aucpr = results['Val']['auc_pr']
train_val_gap = train_aucpr - val_aucpr
print("\n" + "-"*70)
print(f"OVERFITTING DIAGNOSTICS:")
print(f"  Train-Val gap:  {train_val_gap:.3f} (⚠ if > 0.20)")
print(f"  CV AUC-PR:      {cv_aucpr:.3f} (expected close to Val)")
print("-"*70 + "\n")

# Save model (use test scores as reference)
import joblib
model_save_path = Path(processed_dir) / "model" / f"lgb_model_{filename}.joblib"
model_save_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({
    "model": model,
    "cv_auc_pr": cv_aucpr,
    "val_evaluation": results['Val'],
    "study": study
}, model_save_path)
print(f"Saved to: {model_save_path}")

In [ ]:
# get confusion matrix for val set
print("Confusion Matrix (Val):")
print(results['Val']['confusion_matrix'])

### Val and Test 

In [ ]:
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    classification_report, confusion_matrix, precision_recall_curve
)

def val_test_model_evaluate(model, X, y, threshold=None):
    """
    Evaluate a classifier. If threshold is None, finds the optimal threshold
    by maximizing F1 on the precision-recall curve.

    Args:
        model:     Fitted classifier with predict_proba.
        X:         Features DataFrame.
        y:         True labels.
        threshold: Fixed threshold to use. None = find optimal from PR curve.

    Returns:
        Dict with metrics and the threshold used.
    """
    y_pred_proba = model.predict_proba(X)[:, 1]

    # Find optimal threshold by maximizing F1 on PR curve
    if threshold is None:
        precisions, recalls, thresholds = precision_recall_curve(y, y_pred_proba)
        # thresholds is one shorter than precisions/recalls
        f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-8)
        best_idx  = np.argmax(f1_scores)
        threshold = thresholds[best_idx]
        print(f"Optimal threshold: {threshold:.4f}  (max F1={f1_scores[best_idx]:.4f})")

    y_pred = (y_pred_proba >= threshold).astype(int)

    return {
        'threshold':            threshold,
        'auc_roc':              roc_auc_score(y, y_pred_proba),
        'auc_pr':               average_precision_score(y, y_pred_proba),
        'f1':                   f1_scores[best_idx] if threshold is None else f1_score(y, y_pred),
        'classification_report': classification_report(y, y_pred),
        'confusion_matrix':      confusion_matrix(y, y_pred),
    }

In [ ]:
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score,
    precision_recall_curve, classification_report, confusion_matrix
)
def val_test_model_evaluate(model, X, y, threshold=None):
    y_pred_proba = model.predict_proba(X)[:, 1]

    # Always compute PR curve to get optimal threshold and F1
    precisions, recalls, thresholds = precision_recall_curve(y, y_pred_proba)
    f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-8)
    best_idx = np.argmax(f1_scores)
    optimal_threshold = thresholds[best_idx]

    if threshold is None:
        threshold = optimal_threshold
    print(f"Threshold used: {threshold:.4f}  |  Optimal: {optimal_threshold:.4f}  (max F1={f1_scores[best_idx]:.4f})")

    y_pred = (y_pred_proba >= threshold).astype(int)

    return {
        'threshold':             threshold,
        'optimal_threshold':     optimal_threshold,
        'auc_roc':               roc_auc_score(y, y_pred_proba),
        'auc_pr':                average_precision_score(y, y_pred_proba),
        'f1':                    f1_score(y, y_pred),
        'precision':             precision_score(y, y_pred),
        'recall':                recall_score(y, y_pred),
        'classification_report': classification_report(y, y_pred),
        'confusion_matrix':      confusion_matrix(y, y_pred),
    }


In [ ]:
import joblib

# load model and evaluate on validation set and test set
def load_model(max_feature_lag, with_metadata, with_pca=False, features='all', target_future_step=2):
    if with_pca:
        filename = f"apnea_detection_{5*max_feature_lag}secs_lag_features_pc_{features}_{5*target_future_step}secs_future_clean_metadata_{str(with_metadata).lower()}"
    else:
        filename = f"apnea_detection_{5*max_feature_lag}secs_lag_features_{features}_{5*target_future_step}secs_future_clean_metadata_{str(with_metadata).lower()}"

    data = pickle.load(open(Path(processed_dir) / f"{filename}.pkl", "rb"))
    model = joblib.load(Path(processed_dir) / "model" / f"lgb_model_{filename}.joblib")['model']
    return data, model


In [ ]:
val_evaluation = []
for max_feature_lag in [0, 5, 10, 15]:
    data, model = load_model(max_feature_lag=max_feature_lag, with_metadata=True, with_pca=False)
    # evaluate on val and test set
    val_evaluation.append(val_test_model_evaluate(model, data['val_X'], data['val_y']))
    # print classification report and confusion matrix for val set
    print(f"Max Feature Lag: {5*max_feature_lag} secs")
    print("Validation Set Evaluation:")
    print(f"  AUC-ROC={val_evaluation[-1]['auc_roc']:.3f}  AUC-PR={val_evaluation[-1]['auc_pr']:.3f}  F1={val_evaluation[-1]['f1']:.3f}")
    print("  Classification Report:\n", val_evaluation[-1]['classification_report'])
    print("  Confusion Matrix:\n", val_evaluation[-1]['confusion_matrix'])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

max_lags = [0, 5, 10, 15]
auc_prs = [m['auc_pr'] for m in val_evaluation]
macro_f1s = [m['f1'] for m in val_evaluation]
auc_rocs = [m['auc_roc'] for m in val_evaluation]

metrics = {'AUC-PR': auc_prs, 'Macro F1': macro_f1s, 'AUC-ROC': auc_rocs}
x = np.arange(len(metrics))
width = 0.18
lag_labels = [f"{5*lag}s" for lag in max_lags]
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

fig, ax = plt.subplots(figsize=(9, 5))
for i, (label, color) in enumerate(zip(lag_labels, colors)):
    values = [metrics[m][i] for m in metrics]
    ax.bar(x + (i - 1.5) * width, values, width, label=label, color=color)

ax.set_xticks(x)
ax.set_xticklabels(metrics.keys())
ax.set_ylabel("Metric Value")
ax.set_title("Model Performance by Max Feature Lag")
ax.legend(title="Max Lag")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

max_lags = [0, 5, 10, 15]
auc_prs = [m['auc_pr'] for m in val_evaluation]
macro_f1s = [m['f1'] for m in val_evaluation]
auc_rocs = [m['auc_roc'] for m in val_evaluation]

lag_labels = [f"{5*lag}s" for lag in max_lags]
metric_names = ['AUC-PR', 'Macro F1', 'AUC-ROC']
metric_values = [auc_prs, macro_f1s, auc_rocs]
colors = ['#4C72B0', '#DD8452', '#55A868']

x = np.arange(len(lag_labels))
width = 0.25

fig, ax = plt.subplots(figsize=(9, 5))
for i, (name, values, color) in enumerate(zip(metric_names, metric_values, colors)):
    ax.bar(x + (i - 1) * width, values, width, label=name, color=color)

ax.set_xticks(x)
ax.set_xticklabels(lag_labels)
ax.set_xlabel("Max Feature Lag")
ax.set_ylabel("Metric Value")
ax.set_title("Model Performance by Max Feature Lag")
ax.legend(title="Metric")
plt.tight_layout()
plt.show()


In [ ]:
# plot macro-F1, auc-roc, auc-pr vs max feature lag in subplots
import matplotlib.pyplot as plt
max_lags = [0, 5, 10, 15]
macro_f1s = [m['f1'] for m in val_evaluation]
auc_rocs = [m['auc_roc'] for m in val_evaluation]
auc_prs = [m['auc_pr'] for m in val_evaluation]

fig,axs = plt.subplots(1, 3, figsize=(18, 5))
axs[0].plot(max_lags, macro_f1s, marker='o')
axs[0].set_xticks(max_lags)
axs[0].set_xticklabels([f"{5*lag}s" for lag in max_lags])
axs[0].set_xlabel("Max Feature Lag")
axs[0].set_ylabel("Macro F1 Score")
axs[0].set_ylim(0.42, 0.46)

axs[1].plot(max_lags, auc_rocs, marker='o', color='orange')
axs[1].set_xticks(max_lags)
axs[1].set_xticklabels([f"{5*lag}s" for lag in max_lags])
axs[1].set_xlabel("Max Feature Lag")
axs[1].set_ylabel("AUC-ROC")
axs[1].set_ylim(0.79, 0.84)

axs[2].plot(max_lags, auc_prs, marker='o', color='green')
axs[2].set_xticks(max_lags)
axs[2].set_xticklabels([f"{5*lag}s" for lag in max_lags])
axs[2].set_xlabel("Max Feature Lag")
axs[2].set_ylabel("AUC-PR")
axs[2].set_ylim(0.38, 0.44)

In [ ]:
import matplotlib.pyplot as plt

max_lags = [0, 5, 10, 15]
macro_f1s = [m['f1'] for m in val_evaluation]
auc_rocs = [m['auc_roc'] for m in val_evaluation]
auc_prs = [m['auc_pr'] for m in val_evaluation]

lag_labels = [f"{5*lag}s" for lag in max_lags]
FONT = 14

fig, axs = plt.subplots(1, 3, figsize=(18, 5))

for ax, values, color, ylabel, ylim in zip(
    axs,
    [macro_f1s, auc_rocs, auc_prs],
    ['#4C72B0', 'orange', 'green'],
    ['Macro F1 Score', 'AUC-ROC', 'AUC-PR'],
    [(0.42, 0.46), (0.79, 0.84), (0.38, 0.44)],
):
    ax.plot(max_lags, values, marker='o', color=color)
    ax.set_xticks(max_lags)
    ax.set_xticklabels(lag_labels, fontsize=FONT)
    ax.set_xlabel("Max Feature Lag", fontsize=FONT)
    ax.set_ylabel(ylabel, fontsize=FONT)
    ax.set_ylim(ylim)
    ax.tick_params(axis='y', labelsize=FONT)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
final_max_lag = 15

data, model = load_model(max_feature_lag=final_max_lag, with_metadata=True, with_pca=False)
# evaluate on val and test set
test_evaluation = val_test_model_evaluate(model, data['test_X'], data['test_y'])

# print classification report and confusion matrix for val set
print(f"Max Feature Lag: {5*final_max_lag} secs")
print("Test Set Evaluation:")
print(f"  AUC-ROC={test_evaluation['auc_roc']:.3f}  AUC-PR={test_evaluation['auc_pr']:.3f}  F1={test_evaluation['f1']:.3f}")
print("  Classification Report:\n", test_evaluation['classification_report'])
print("  Confusion Matrix:\n", test_evaluation['confusion_matrix'])

In [ ]:
import matplotlib.pyplot as plt

max_lags = [0, 5, 10, 15]
macro_f1s = [m['f1'] for m in val_evaluation]
auc_rocs = [m['auc_roc'] for m in val_evaluation]
auc_prs = [m['auc_pr'] for m in val_evaluation]

lag_labels = [f"{5*lag}" for lag in max_lags]
FONT = 16

fig, axs = plt.subplots(1, 3, figsize=(18, 5.5))

for ax, values, color, ylabel, ylim in zip(
    axs,
    [macro_f1s, auc_rocs, auc_prs],
    ['#4C72B0', 'orange', 'green'],
    ['Macro F1 Score', 'AUC-ROC', 'AUC-PR'],
    [(0.42, 0.46), (0.79, 0.84), (0.38, 0.44)],
):
    ax.plot(max_lags, values, marker='o', color=color, linewidth=3, markersize=12)
    ax.set_xticks(max_lags)
    ax.set_xticklabels(lag_labels, fontsize=FONT)
    ax.set_xlabel("Max Feature Lag (in secs)", fontsize=FONT+4)
    ax.set_ylabel("Score", fontsize=FONT+4)
    ax.set_title(f"{ylabel}", fontsize=FONT+6)
    ax.set_ylim(ylim)
    ax.tick_params(axis='y', labelsize=FONT)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.2f}"))
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()
